# Voice2Reel — Mascot V1 + trained Qwen Story Director
**الافتراضي الآن:** الشخصية الأصلية Mascot V1 هي SVG بمفاصل مترابطة حقيقيًا (nested rig). يتم تحريكها في كل إطار بدون إعادة إنشاء خطوط الأذرع، ثم إنتاج Reels بدقة 1080×1920. تدريب Qwen للسيناريو القديم محفوظ في Drive، ولا يعاد افتراضيًا.

**الترتيب:** إعدادات الفيديو → Setup → معاينة الشخصية الجديدة → (تدريب Qwen للـSVG اختياري) → (تدريب السيناريو اختياري) → توليد القصة → الصوت → رندر الفيديو.

**ملاحظة:** خيار qwen_static يستخدم صورة ثابتة ينتجها Adapter منفصل؛ لتحريك المفاصل فعليًا استخدم CHARACTER_ENGINE=rig. لا يحتاج Rig إلى إعادة تدريب Qwen، لكنه لا يجعل Qwen مُنشئًا للشخصية الأساسية. إعدادات التدريب منفصلة حتى لا تستهلك GPU بالخطأ.


## 1 — إعدادات الفكرة والشخصية

In [ ]:
#@title 🎬 إعدادات الفيديو
IDEA = "اشرح كيف يعمل الراوتر وكيف يتصل بالإنترنت ويوزع الواي فاي" #@param {type:"string"}
LANGUAGE = "ar" #@param ["ar","en"]
CHARACTER = "classic" #@param ["classic"]
RUN_TTS = True #@param {type:"boolean"}
RUN_RENDER = True #@param {type:"boolean"}
CHARACTER_ENGINE = "rig" #@param ["rig","qwen_static","legacy"]
RUN_QWEN_MASCOT = False #@param {type:"boolean"}
RUN_TRAINING = False #@param {type:"boolean"}
DATASET_COUNT = 3000 #@param {type:"integer"}
EPOCHS = 3 #@param {type:"integer"}
PATIENCE = 2 #@param {type:"integer"}
ADAPTER_OVERRIDE = "" #@param {type:"string"}
assert IDEA.strip() and len(IDEA) <= 2000
assert 1 <= DATASET_COUNT <= 2000000
print("💡", IDEA)

## 2 — تثبيت البيئة وربط Drive (مرة واحدة لكل جلسة)

In [ ]:
#@title Setup
import os, sys, subprocess, hashlib
from pathlib import Path
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")
REPO = Path("/content/Voice2Reel")
STORE = Path("/content/drive/MyDrive/Voice2Reel_09")
STORE.mkdir(parents=True, exist_ok=True)
if not (REPO / ".git").exists():
    subprocess.run(["git","clone","--branch","test-new-data",
                    "https://github.com/garaahmad/Voice2Reel.git",str(REPO)],check=True)
else:
    subprocess.run(["git","pull","--ff-only","origin","test-new-data"],cwd=str(REPO),check=True)
def run(*args, cwd=REPO, timeout=None):
    cmd=list(map(str,args))
    print("▶", " ".join(cmd), flush=True)
    subprocess.run(cmd,cwd=str(cwd),check=True,timeout=timeout)
run(sys.executable,"-m","pip","install","-q","-r","requirements/character-lab.txt")
run(sys.executable,"-m","pip","install","-q","-e",".","edge-tts","cairosvg")
# Inference needs these even when both training switches are False.
run(sys.executable,"-m","pip","install","-q",
    "transformers==4.57.3","peft==0.17.1","accelerate==1.10.1")
sys.path.insert(0,str(REPO))
from voice2reel.character_templates import lock_template,load_lock
from voice2reel.mascot_animator import MascotAnimator,render_pose_sheet
from voice2reel.production_contract import ProductionPlan
# Each idea gets an isolated directory; changing the idea cannot reuse stale audio.
job_id = hashlib.sha256((IDEA+"|"+CHARACTER+"|"+LANGUAGE).encode()).hexdigest()[:12]
RESULTS = STORE/"ideas"/job_id
RESULTS.mkdir(parents=True,exist_ok=True)
DATA = STORE/"dataset"
MODEL_DIR = STORE/"qwen_adapter_run"
LOCK = RESULTS/"character_lock.json"
PLAN_PATH = RESULTS/"scene.json"
VOICE = RESULTS/"voice.mp3"
VIDEO = RESULTS/"final.mp4"
MUSIC = RESULTS/"music.mp3"
lock_template(CHARACTER, LOCK)
display(MascotAnimator().frame_image("idle", 1.25, 3, width=400))
print("📁", RESULTS)

## Mascot V1 — تحقق من الرسم والمفاصل قبل إنتاج الفيديو


In [ ]:
#@title ✅ Preview mascot_v1 poses before expensive rendering
from IPython.display import display
from PIL import Image
PREVIEW_A = RESULTS / "mascot_v1_poses_1.png"
PREVIEW_B = RESULTS / "mascot_v1_poses_2.png"
render_pose_sheet(PREVIEW_A, ("idle", "present", "point", "wave"))
render_pose_sheet(PREVIEW_B, ("think", "explain", "celebrate", "step"))
display(Image.open(PREVIEW_A))
display(Image.open(PREVIEW_B))
print("✅ Original jointed mascot preview:", PREVIEW_A, PREVIEW_B)


## 3 — 🎨 تدريب Qwen لإنشاء شخصية SVG جديدة
هذا **Adapter منفصل** عن Adapter السيناريو. Qwen هنا يولّد رسم SVG متجهي (ثم PNG)، وليس نموذج انتشار يولّد صورًا فوتوغرافية. البيانات صناعية مولّدة من قوالب؛ الجودة تحتاج مراجعة بصرية وتوسيع بيانات بشرية لاحقًا. تدريب GPU يتم فقط عندما RUN_MASCOT_TRAINING=True. لا تشغّل خلية تدريب السيناريو إن كان لديك Adapter قديم صالح.

In [ ]:
#@title 🎨 Generate mascot SVG dataset and train separate Qwen adapter
import time
RUN_MASCOT_TRAINING = False #@param {type:"boolean"}
MASCOT_DATASET_COUNT = 1200 #@param {type:"integer"}
MASCOT_EPOCHS = 2 #@param {type:"integer"}
MASCOT_ADAPTER_OVERRIDE = "" #@param {type:"string"}
MASCOT_DATA = STORE / "mascot_svg_dataset"
MASCOT_LATEST = STORE / "latest_mascot_adapter.txt"
run(sys.executable, "-m", "pip", "install", "-q", "cairosvg")
if RUN_MASCOT_TRAINING:
    run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/text.txt")
    run(sys.executable, "-m", "scripts.build_mascot_svg_dataset",
        "--output", MASCOT_DATA, "--count", MASCOT_DATASET_COUNT)
    MASCOT_TRAIN_OUT = STORE / ("qwen_mascot_svg_" + time.strftime("%Y%m%d_%H%M%S"))
    run(sys.executable, "-u", "-m", "voice2reel.train_mascot_svg",
        "--train", MASCOT_DATA / "train.jsonl",
        "--validation", MASCOT_DATA / "validation.jsonl",
        "--output", MASCOT_TRAIN_OUT,
        "--epochs", MASCOT_EPOCHS, timeout=14400)
    MASCOT_ADAPTER_OVERRIDE = str(MASCOT_TRAIN_OUT / "adapter")
    MASCOT_LATEST.write_text(MASCOT_ADAPTER_OVERRIDE, encoding="utf-8")
    print("✅ New mascot adapter:", MASCOT_ADAPTER_OVERRIDE)
else:
    print("Using existing mascot adapter (no training).")


In [ ]:
#@title 🖌️ Ask trained Qwen to draw a new mascot (SVG + PNG)
from IPython.display import Image as NotebookImage, display
MASCOT_PROMPT = "Draw an original happy charcoal stickman mascot, explaining pose, ink #161c28, background #fefbf5, accent #ff6955. Rounded hands and feet, smooth silhouette, expressive face, no hat." #@param {type:"string"}
if RUN_QWEN_MASCOT:
    choices = []
    if MASCOT_ADAPTER_OVERRIDE.strip():
        choices.append(Path(MASCOT_ADAPTER_OVERRIDE.strip()))
    if MASCOT_LATEST.is_file():
        choices.append(Path(MASCOT_LATEST.read_text(encoding="utf-8").strip()))
    choices.extend(sorted(STORE.glob("qwen_mascot_svg_*/adapter"), reverse=True))
    MASCOT_ADAPTER = next((p for p in choices if (p / "adapter_config.json").is_file()), None)
    if MASCOT_ADAPTER is None:
        raise FileNotFoundError("No trained mascot adapter. Train above or set MASCOT_ADAPTER_OVERRIDE.")
    MASCOT_SVG = RESULTS / "qwen_mascot.svg"
    MASCOT_PNG = RESULTS / "qwen_mascot.png"
    MASCOT_SVG.unlink(missing_ok=True)
    MASCOT_PNG.unlink(missing_ok=True)
    run(sys.executable, "-u", "-m", "voice2reel.generate_mascot_svg",
        "--prompt", MASCOT_PROMPT, "--adapter", MASCOT_ADAPTER,
        "--svg", MASCOT_SVG, "--png", MASCOT_PNG, timeout=1200)
    display(NotebookImage(filename=str(MASCOT_PNG)))
    print("🎨 Saved:", MASCOT_SVG, MASCOT_PNG)
    print("Next: train/use scene adapter and continue with story, voice and video cells.")

else:
    print("⏭ Qwen static SVG generation skipped. Mascot V1 rig remains active.")


## 3 — إعداد بيانات التدريب وتدريب Qwen (اختياري، عند الحاجة فقط)
إذا لديك Adapter مدرب حديثًا، اترك RUN_TRAINING=False. توليد بيانات جديدة وحده لا يدرّب الموديل. لا تستخدم Adapter قديمًا وتتوقع منه معرفة العناصر الجديدة.

In [ ]:
#@title 🏋️ Generate dataset and train Qwen QLoRA (first run only)
import json, time
if RUN_TRAINING:
    run(sys.executable,'-m','pip','install','-q','-r','requirements/text.txt')
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'],check=True)
    check='import torch; from transformers import Trainer, AutoModelForCausalLM; assert torch.cuda.is_available()'
    probe=subprocess.run([sys.executable,'-c',check],capture_output=True,text=True)
    if probe.returncode:
        print('Torch import failed:',probe.stderr[-3000:])
        run(sys.executable,'-m','pip','uninstall','-y','torch','torchvision','torchaudio')
        run(sys.executable,'-m','pip','install','--no-cache-dir','torch==2.8.0')
        probe=subprocess.run([sys.executable,'-c',check],capture_output=True,text=True)
        if probe.returncode:
            print(probe.stderr[-5000:])
            raise RuntimeError('PyTorch still inconsistent. Disconnect and delete Colab runtime, reopen, then Run all.')
    print('Training imports and CUDA verified')
    DATA.mkdir(parents=True,exist_ok=True)
    run(sys.executable,"-m","scripts.build_large_scene_curriculum",
        "--count",DATASET_COUNT,"--output",DATA,
        "--check-every",max(1,min(DATASET_COUNT,1000)))
    for split in ("train","validation","test"):
        path=DATA/(split+".jsonl")
        if not path.is_file() or path.stat().st_size==0:
            raise RuntimeError("Missing training split: "+str(path))
    # Always write into a NEW empty directory; trainer rejects non-empty output.
    stamp=time.strftime("%Y%m%d_%H%M%S")
    train_output=STORE/("qwen_adapter_scene_"+stamp)
    run(sys.executable,"-u","-m","voice2reel.train_python_scene",
        "--train",DATA/"train.jsonl","--validation",DATA/"validation.jsonl",
        "--output",train_output,"--epochs",EPOCHS,"--patience",PATIENCE,timeout=21600)
    trained_adapter=train_output/"adapter"
    if not (trained_adapter/"adapter_config.json").is_file():
        raise RuntimeError("Training finished without a valid adapter_config.json")
    ADAPTER_OVERRIDE=str(trained_adapter)
    (STORE/"latest_scene_adapter.txt").write_text(ADAPTER_OVERRIDE,encoding="utf-8")
    print("✅ Adapter saved permanently:",ADAPTER_OVERRIDE)
    print("💡 For future ideas set RUN_TRAINING=False and change IDEA.")
else:
    print("⏭ Training skipped; will reuse an existing adapter.")


## 4 — توليد السيناريو بواسطة Qwen من IDEA
**مهم:** لا يوجد fallback ثابت. عند فشل النموذج تظهر الرسالة الحقيقية. سيُحفظ نص DSL بجانب JSON لمراجعته.

In [ ]:
#@title 🧠 Generate story with trained Qwen
import json, sys, subprocess
from pathlib import Path

# Use your completed adapter; do NOT start another training run.
RUN_TRAINING = False
ADAPTER_OVERRIDE = "/content/drive/MyDrive/Voice2Reel_09/qwen_adapter_scene_20261116_131641/adapter"

# Refresh the generation code in the already-cloned Colab repository.
subprocess.run(["git", "pull", "--ff-only", "origin", "test-new-data"],
               cwd=str(REPO), check=True)

candidates = []
if ADAPTER_OVERRIDE.strip():
    candidates.append(Path(ADAPTER_OVERRIDE.strip()))
latest = STORE / "latest_scene_adapter.txt"
if latest.is_file():
    candidates.append(Path(latest.read_text(encoding="utf-8").strip()))
candidates.extend(sorted(STORE.glob("qwen_adapter_scene_*/adapter"), reverse=True))
ADAPTER = next((p for p in candidates if (p / "adapter_config.json").is_file()), None)
if ADAPTER is None:
    raise FileNotFoundError("No valid scene adapter found. Check ADAPTER_OVERRIDE.")
print("🧠 Using trained adapter:", ADAPTER)

# Never render stale scene data after a generation failure.
PLAN_PATH.unlink(missing_ok=True)
cmd = [sys.executable, "-u", "-m", "voice2reel.generate_python_scene",
       "--idea", IDEA, "--lock", str(LOCK), "--adapter", str(ADAPTER),
       "--output", str(PLAN_PATH), "--max-attempts", "3"]
proc = subprocess.run(cmd, cwd=str(REPO), text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=1200)
print(proc.stdout, flush=True)
if proc.returncode:
    raise RuntimeError("Qwen failed after validation retries; see output above.")
plan = ProductionPlan.model_validate_json(PLAN_PATH.read_text(encoding="utf-8"))
plan = plan.model_copy(update={"width": 1080, "height": 1920})
PLAN_PATH.write_text(plan.model_dump_json(indent=2), encoding="utf-8")
print("🎞️ New storyboard:", len(plan.shots), "shots")
for i, shot in enumerate(plan.shots, 1):
    print(i, shot.seconds, "s |", shot.narration,
          "| props:", [o.kind for o in shot.objects])


## 5 — تحسين الصوت العربي فقط
الآن يوجد اختيار أصوات عربية متعددة وسرعة وصعود نغمة اختياريان. يمكنك الاستماع قبل الرندر، وتعديل كلام كل مشهد داخل VOICE_OVERRIDES، وتصحيح نطق المصطلحات عبر VOICE_TERMS؛ المشاهد والشخصية لا تتغير.


### 🎧 جرّب الأصوات العربية قبل إنتاج الفيديو
هذه الخلية اختيارية. شغّلها لمقارنة الأصوات باستخدام نفس الجملة من السيناريو. لا تغيّر أي شخصية أو حركة.


In [ ]:
#@title 🎧 Compare Arabic voices before making the full video (optional)
MAKE_VOICE_PREVIEWS = False #@param {type:"boolean"}
if MAKE_VOICE_PREVIEWS:
    from IPython.display import Audio, display
    import subprocess
    subprocess.run(["git", "pull", "--ff-only", "origin", "test-new-data"],
                   cwd=str(REPO), check=True)
    if LANGUAGE != "ar":
        raise ValueError("Arabic voice comparison is available when LANGUAGE='ar'")
    plan = ProductionPlan.model_validate_json(PLAN_PATH.read_text(encoding="utf-8"))
    sample = plan.shots[0].narration.strip()
    if len(sample) > 200:
        sample = sample[:200].rsplit(" ", 1)[0]
    preview_voices = {
        "تيم - الأردن / قريب من الشامي": "ar-JO-TaimNeural",
        "شاكر - مصر": "ar-EG-ShakirNeural",
        "حامد - السعودية": "ar-SA-HamedNeural",
        "زارية - السعودية (صوت أنثوي)": "ar-SA-ZariyahNeural",
    }
    folder = RESULTS / "voice_comparison"
    folder.mkdir(parents=True, exist_ok=True)
    for label, voice_id in preview_voices.items():
        path = folder / f"{voice_id}.mp3"
        cmd = [
            sys.executable, "-u", "-m", "voice2reel.production_voice",
            "--preview-text", sample, "--language", "ar", "--voice", voice_id,
            "--rate=+0%", "--pitch=+0Hz", "--output", str(path)
        ]
        result = subprocess.run(cmd, cwd=str(REPO), capture_output=True,
                                text=True, timeout=180)
        if result.returncode:
            print("❌", label, result.stdout[-1000:], result.stderr[-1000:])
            continue
        print("🎙️", label, "—", voice_id)
        display(Audio(filename=str(path)))
    print("اختَر الصوت الأنسب، واكتب الـvoice ID في خلية Voiceover.")
else:
    print("⏭ Voice comparison skipped. Enable MAKE_VOICE_PREVIEWS to audition voices.")


In [ ]:
#@title 🔊 Voiceover — natural Arabic options (audio only)
VOICE_NAME = "ar-JO-TaimNeural" #@param ["ar-JO-TaimNeural","ar-EG-ShakirNeural","ar-SA-HamedNeural","ar-SA-ZariyahNeural","ar-EG-SalmaNeural","en-US-GuyNeural","en-US-JennyNeural"]
VOICE_RATE = "+0%" #@param ["-15%","-10%","-8%","-5%","-3%","+0%","+3%","+5%"]
VOICE_PITCH = "+0Hz" #@param ["-4Hz","-2Hz","+0Hz","+2Hz","+4Hz"]
# Optional speech-only overrides: 1-based shot numbering; scene.json remains unchanged.
# Example: VOICE_OVERRIDES = {"1": "أهلًا فيكم! اليوم رح نحكي عن الراوتر."}
VOICE_OVERRIDES = {}
# Optional pronunciation for code names, applied only to the spoken narration.
# Example: VOICE_TERMS = {"Flutter": "فلاتَر", "setState": "سِت ستيت"}
VOICE_TERMS = {}
if RUN_TTS:
    import json, subprocess, re
    from IPython.display import Audio, display
    subprocess.run(["git", "pull", "--ff-only", "origin", "test-new-data"],
                   cwd=str(REPO), check=True)
    options = []
    if VOICE_OVERRIDES:
        override_path = RESULTS / "voice_only_overrides.json"
        override_path.write_text(
            json.dumps({"shots": {str(k): v for k, v in VOICE_OVERRIDES.items()}},
                       ensure_ascii=False, indent=2), encoding="utf-8")
        options.extend(["--script-json", str(override_path)])
    if VOICE_TERMS:
        terms_path = RESULTS / "voice_only_terms.json"
        terms_path.write_text(
            json.dumps(VOICE_TERMS, ensure_ascii=False, indent=2),
            encoding="utf-8")
        options.extend(["--terms-json", str(terms_path)])
    if VOICE.is_file():
        VOICE.unlink()
    selected_voice = VOICE_NAME if LANGUAGE == "ar" else "en-US-GuyNeural"
    run(sys.executable, "-u", "-m", "voice2reel.production_voice",
        "--plan", PLAN_PATH, "--output", VOICE, "--language", LANGUAGE,
        "--voice", selected_voice,
        f"--rate={VOICE_RATE}", f"--pitch={VOICE_PITCH}",
        *options, timeout=600)
    if not VOICE.is_file() or VOICE.stat().st_size == 0:
        raise RuntimeError("TTS did not produce usable audio.")
    print("🎧 Voice preview (before rendering):")
    display(Audio(filename=str(VOICE)))
    script_path = VOICE.with_suffix(".speech.txt")
    if script_path.exists():
        print("📝 نص الكلام الذي قُرئ:")
        print(script_path.read_text(encoding="utf-8"))
    # Report when the voice would be truncated by the current video duration.
    import imageio_ffmpeg
    check = subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-hide_banner",
                            "-i", str(VOICE)], capture_output=True, text=True)
    match = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", check.stderr)
    if match:
        audio_seconds = int(match[1])*3600 + int(match[2])*60 + float(match[3])
        planned_seconds = sum(x.seconds for x in ProductionPlan.model_validate_json(
            PLAN_PATH.read_text(encoding="utf-8")).shots)
        print(f"🔊 Audio: {audio_seconds:.1f}s | 🎥 Video: {planned_seconds:.1f}s")
        if audio_seconds > planned_seconds + 0.2:
            print("⚠️ الصوت أطول من الفيديو وسيُقطع آخره. اختصر النص في VOICE_OVERRIDES أو عدّل طول السيناريو لاحقًا.")
else:
    print("⏭ Narration disabled.")


## 6 — رندر الفيديو وعرضه

In [ ]:
#@title 🎥 Render Instagram Reels MP4 using Mascot V1 rig
from IPython.display import Video,display
# Fetch renderer and character improvements before rendering.
subprocess.run(["git", "pull", "--ff-only", "origin", "test-new-data"],
               cwd=str(REPO), check=True)
plan = ProductionPlan.model_validate_json(PLAN_PATH.read_text(encoding="utf-8"))
plan = plan.model_copy(update={"width": 1080, "height": 1920})
PLAN_PATH.write_text(plan.model_dump_json(indent=2), encoding="utf-8")
if RUN_RENDER:
    cmd=[sys.executable,"-u","-m","voice2reel.production_pipeline",
         "--plan",str(PLAN_PATH),"--lock",str(LOCK),"--output",str(VIDEO)]
    cmd += ["--character-engine", CHARACTER_ENGINE]
    if CHARACTER_ENGINE == "qwen_static":
        if not RUN_QWEN_MASCOT or not MASCOT_PNG.is_file():
            raise FileNotFoundError("Enable RUN_QWEN_MASCOT and generate a Qwen mascot PNG first.")
        cmd += ["--mascot-png", str(MASCOT_PNG)]
    if RUN_TTS:
        if not VOICE.is_file() or VOICE.stat().st_size==0:
            raise FileNotFoundError("Run voiceover cell before rendering.")
        cmd+=["--voice",str(VOICE)]
    if MUSIC.is_file():cmd+=["--music",str(MUSIC)]
    p=subprocess.run(cmd,cwd=str(REPO),text=True,stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT,timeout=3600)
    print(p.stdout,flush=True)
    if p.returncode:raise RuntimeError("Rendering failed; see log above.")
    if not VIDEO.is_file() or VIDEO.stat().st_size==0:
        raise RuntimeError("Missing or empty video.")
    display(Video(str(VIDEO),embed=False,width=360))
    print("🎬",VIDEO,"MB:",round(VIDEO.stat().st_size/1e6,2))
else:
    print("⏭ Rendering disabled")